# Player $\theta$ evaluation

Held-out **action prediction** with vs without per-player tilt $\hat\theta$ from `artifacts/player_thetas.json`.

**Splits:** `sessions_theta.txt` (EM) and `sessions_filter.txt` (online). Helpers: `utils.eval.player_thetas_evaluation_helpers`, `utils.eval.common`.


In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

def _find_repo_root() -> Path:
    p = Path.cwd().resolve()
    for x in (p, *p.parents):
        if (x / "runners" / "common.py").is_file():
            return x
    raise FileNotFoundError("Run from inside the repo (runners/common.py not found).")

REPO = _find_repo_root()
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

import json

import matplotlib.pyplot as plt
import numpy as np

from utils.eval.common import em_and_online_refs
from utils.eval.player_thetas_evaluation_helpers import (
    load_global_betas,
    load_player_thetas,
    build_player_prediction_dict,
    flip_rate,
    realised_shifts,
    heldout_gradient_norm_components,
    online_summary_rows,
    nll,
    mean_brier,
)

priors = json.loads((REPO / "artifacts" / "global_priors.json").read_text())
BETA_PRE, BETA_FACING, BETA_NO_BET = load_global_betas(priors)
thetas_payload = json.loads((REPO / "artifacts" / "player_thetas.json").read_text())
PLAYERS, THETA_PRE, THETA_POST = load_player_thetas(thetas_payload)
print("players:", PLAYERS)


In [ ]:
em_refs, online_refs, em_s, online_s = em_and_online_refs(REPO)
print("theta sessions:", em_s)
print("filter sessions:", online_s)
print("hands em", len(em_refs), "online", len(online_refs))

In [ ]:
PRED = build_player_prediction_dict(
    PLAYERS,
    B_pre=BETA_PRE,
    B_facing=BETA_FACING,
    B_no_bet=BETA_NO_BET,
    theta_pre=THETA_PRE,
    theta_post=THETA_POST,
    em_refs=em_refs,
    online_refs=online_refs,
)
print("buckets", len(PRED))

In [ ]:
print(f"{'player':<10} {'split':<7} {'head':<8} {'N':>4} {'NLL_0':>8} {'NLL_θ':>8} {'Δ':>8}")
for (p, sp, head), (P0, Pt, y) in sorted(PRED.items()):
    n0, nt = nll(P0, y), nll(Pt, y)
    print(f"{p:<10} {sp:<7} {head:<8} {y.size:>4} {n0:>8.4f} {nt:>8.4f} {n0-nt:>+8.4f}")

In [ ]:
print(f"{'player':<10} {'split':<7} {'head':<8} {'N':>4} {'Brier_0':>9} {'Brier_θ':>9} {'Δ':>9}")
for (p, sp, head), (P0, Pt, y) in sorted(PRED.items()):
    b0, bt = mean_brier(P0, y), mean_brier(Pt, y)
    print(f"{p:<10} {sp:<7} {head:<8} {y.size:>4} {b0:>9.4f} {bt:>9.4f} {b0-bt:>+9.4f}")

In [ ]:
print(f"{'player':<10} {'split':<7} {'head':<8} {'N':>4} {'acc_0':>7} {'acc_θ':>7} {'flips':>7}")
for (p, sp, head), (P0, Pt, y) in sorted(PRED.items()):
    acc0 = float((P0.argmax(axis=1) == y.astype(int)).mean()) if y.size else float("nan")
    acct = float((Pt.argmax(axis=1) == y.astype(int)).mean()) if y.size else float("nan")
    print(f"{p:<10} {sp:<7} {head:<8} {y.size:>4} {acc0:>7.3f} {acct:>7.3f} {flip_rate(P0, Pt):>7.3f}")

In [ ]:
shift_arrays = {k: realised_shifts(P0, Pt, y) for k, (P0, Pt, y) in PRED.items()}
print(f"{'player':<10} {'split':<7} {'head':<8} {'N':>4} {'mean|Δ|':>9} {'median|Δ|':>11}")
for (p, sp, head), (P0, Pt, y) in sorted(PRED.items()):
    s = shift_arrays[(p, sp, head)]
    if s.size == 0:
        print(f"{p:<10} {sp:<7} {head:<8} {0:>4}")
        continue
    print(f"{p:<10} {sp:<7} {head:<8} {s.size:>4} {s.mean():>9.4f} {np.median(s):>11.4f}")

In [ ]:
print(f"{'player':<10} {'split':<7} {'head':<8} {'||g||':>9}")
for (p, sp, head), (P0, Pt, y) in sorted(PRED.items()):
    if head == "preflop":
        th, k = THETA_PRE[p], 3
    elif head == "facing":
        th, k = THETA_POST[p], 3
    else:
        th, k = THETA_POST[p], 2
    g = heldout_gradient_norm_components(Pt, y, th, k)
    print(f"{p:<10} {sp:<7} {head:<8} {float(np.linalg.norm(g)):>9.5f}")

In [ ]:
rows_summary = online_summary_rows(PRED, players=PLAYERS, theta_pre=THETA_PRE, theta_post=THETA_POST)
for r in rows_summary:
    print(r)

In [ ]:
fig, axes = plt.subplots(len(PLAYERS), 3, figsize=(13, 3.5 * len(PLAYERS)), sharex=True)
if len(PLAYERS) == 1:
    axes = np.array([axes])
for i, p in enumerate(PLAYERS):
    for j, head in enumerate(("preflop", "facing", "no_bet")):
        ax = axes[i, j]
        s = shift_arrays[(p, "online", head)]
        if s.size == 0:
            ax.set_title(f"{p} {head} (no rows)")
            continue
        ax.hist(s, bins=20, edgecolor="black")
        ax.set_title(f"{p} — {head} (online, n={s.size})")
        ax.set_xlabel("|ΔP(a★)|")
        ax.set_ylabel("rows")
        ax.grid(alpha=0.3)
fig.suptitle("|ΔP(realised action)| under θ̂ vs θ=0 (online)")
fig.tight_layout()
plt.show()